(application)=
# Application runtime

You can use the {py:meth}`~mlrun.runtimes.ApplicationRuntime` to provide an image that runs on top of your deployed model. 

The application runtime deploys a container image (for example, a web application) that is exposed on a specific port, and a command to run the HTTP server. The runtime is based on top of Nuclio, and adds the application as a side-car to a Nuclio function pod while the actual function is a reverse proxy to that application. 

You can set an existing image to run in the application, or let the application runtime build the side-car image for you, by specifying the source code, and pulling at build-time. 

> Note: The default base image is python:3.11 (or 3.9 based on the client python version) when not specifying an image for the application.
    
An API Gateway, by default, is in front of the application and can provide different authentication methods, or none.

Typical use cases are:
- Deploy a [Vizro](https://github.com/mckinsey/vizro) dashboard that communicates with an external source (for example, a serving model) to display graphs, data, and inference.
- Deploy a model and a UI &mdash; the model serving is the backend and the UI is the side car.
- Deploy a fastapi web-server with an MLRun model. In this case, the Nuclio function is a reverse proxy and the user web-app is the side car.

```{admonition} Note
When using an application runtime with min-replicas set to 0 (Scale-To-Zero enabled) and direct port access enabled , be aware of a known limitation in metric collection. Scale-to-Zero currently relies on metrics collected only from the main Nuclio container that runs the reverse proxy. Traffic sent directly to the application runtime sidecar bypasses these metrics, so such requests are not counted. As a result, the function may scale down to zero even while it is actively serving traffic through the direct port. In addition, once scaled to zero, the function can only be scaled back up by invoking the main Nuclio port, not the direct port. To avoid unexpected behavior, do not rely on direct port traffic alone when using Scale-To-Zero.
```

**In this section**
- [Usage examples](#usage-examples)
- [Authentication modes](#authentication-modes)
- [Expose multiple ports](#expose-multiple-ports)
- [Application runtime in a dark environment](#application-runtime-in-a-dark-environment)
- [Application and serving function integration](#application-and-serving-function-integration)

## Usage examples

Deploy a Vizro dashboard from a pre-built image:

In [ ]:
# Create an application runtime (with pre-built image)
application = project.set_function(
    name="my-vizro-dashboard", kind="application", image="repo/my-vizro-image:latest"
)
# Set the port that the side-car listens on
application.set_internal_application_port(port=8050)

# Deploy
application.deploy()

Deploy a Vizro dashboard from a source archive or git:

In [ ]:
# Specify the source to be loaded at build-time or run-time
application = project.set_function(
    name="my-vizro-dashboard", kind="application", requirements=["vizro"]
)
application.set_internal_application_port(8050)
application.spec.command = "gunicorn"
application.spec.args = [
    "<my-app>:<my-server>",
    "--bind",
    "0.0.0.0:8050",
]

# Provide code artifacts
application.with_source_archive(
    "git://github.com/org/repo#my-branch", pull_at_runtime=False
)

# Build the application image via MLRun and deploy the Nuclio function
# Optionally add mlrun
application.deploy(with_mlrun=False)

Reusing an already built reverse proxy image is done when:
- Redeploying a function that built the reverse proxy once and has `application.status.container_image` enriched.
- It was already built manually with `mlrun.runtimes.ApplicationRuntime.deploy_reverse_proxy_image()`.

Using one of the above options can help minimize redundant builds and streamline the development cycle.

## Authentication modes

An application runtime can be accessed through an API Gateway that supports various authentication methods. 

The default authentication mode is `none` for open source and `access-key` for the Iguazio platform.

The different authentication modes can be configured as follows (see {py:meth}`~mlrun.runtimes.nuclio.api_gateway.APIGateway` for further information):

In [ ]:
from mlrun.common.schemas.api_gateway import APIGatewayAuthenticationMode

# Unless disabled, the default API gateway is created when the application is deployed
application.deploy(create_default_api_gateway=False)

# Create API gateway without authentication
application.create_api_gateway(
    authentication_mode=APIGatewayAuthenticationMode.none,
)

# Basic authentication mode.
# This means that the application can be invoked only using the provided credentials
application.create_api_gateway(
    authentication_mode=APIGatewayAuthenticationMode.basic,
    authentication_creds=("my-username", "my-password"),
)

# Access-key authentication mode. the application can be invoked only with a valid session
application.create_api_gateway(
    authentication_mode=APIGatewayAuthenticationMode.access_key,
)

### API gateway configurations

If you want to deploy an application with the default API Gateway, simply run `app.deploy()`. 
If you don’t need the default API Gateway, or if you prefer to create your own custom one, set `create_default_api_gateway=False` when calling `deploy()` and then manually create a [custom API Gateway](../concepts/nuclio-real-time-functions.ipynb).
If you specify a custom port, you must set `direct_port_access=True`; otherwise, the value is ignored and the internal application port is used instead.

There are additional parameters that can be configured when creating an API gateway:

In [ ]:
application.create_api_gateway(
    # The name of the API gateway
    name="my-api-gateway",
    # Optional path of the API gateway, default value is "/". The given path should be supported by the deployed application
    path="/",
    # Set to True to allow direct port access to the application sidecar
    direct_port_access=False,
    # Set to True to force SSL redirect, False to disable. Defaults to mlrun.mlconf.force_api_gateway_ssl_redirect()
    ssl_redirect=True,
    # Set the API gateway as the default for the application (`status.api_gateway`)
    set_as_default=False,
)

## Expose multiple ports

```{admonition} Note
Requires Nuclio 1.14.14 and higher.
```

By default, MLRun creates an API gateway for each application that forwards the events to the internal port, in this example 8010.
Some applications use different ports, for example, for the application API and the dashboard. 
This example uses the default port 8010 for the application and adds port 8020 for other uses.

```
# Enable direct_port_access
direct_port_access=True
# Set multiple ports
app.with_sidecar(ports=[8010,8020])
# Set the internal (main) port
app.set_internal_application_port(8010)
# Add API gateway for second port
url_direct = app.create_api_gateway(port=8020,name="port-8020",direct_port_access=True,authentication_mode=mlrun.common.schemas.api_gateway.APIGatewayAuthenticationMode.none)
```

## Application runtime in a dark environment

To use application runtime in a dark (air-gapped) environment, you need to build the reverse proxy image and push it to a private registry, following the steps below:

In [ ]:
# 1. Create the reverse proxy image in a non air-gapped system
import mlrun.runtimes

mlrun.runtimes.ApplicationRuntime.deploy_reverse_proxy_image()

# 2. The created image name is saved on the ApplicationRuntime class:
mlrun.runtimes.ApplicationRuntime.reverse_proxy_image

# 3. Push the created image to the system’s docker registry

# 4. On the air-gapped environment, set the image on the class property:
mlrun.runtimes.ApplicationRuntime.reverse_proxy_image = (
    "registry/reverse-proxy-image:<tag>"
)

# 5. When creating application functions, this image will be used as the reverse proxy image, and it won’t be built again.

## Application and serving function integration
This example demonstrates deploying a serving function and using it with the application.
</br></br>
<u>Serving creation</u>:</br>
First, create the model file and save it as a .py file.

In [ ]:
%%writefile add_ten_model.py

from mlrun.serving import V2ModelServer

class AddTenModel(V2ModelServer):
    def load(self):
        # No actual model to load, just a demo
        pass

    def predict(self, request):
        input = request['inputs'][0]
        result = input + 10
        return {"outputs": [result]}

Now, deploy this model as a serving function:

In [ ]:
import mlrun

project_name = "app-demo-flask"
model_name = "add-ten-model"
model_path = "add_ten_model.py"

project = mlrun.get_or_create_project(project_name)

In [ ]:
# Create the serving function
# Create the serving function
function = project.set_function(
    name="add-ten-serving",
    kind="serving",
    func=model_path,
)
project.save()

In [ ]:
# Add the model to the function (even though there's no real model in this case)
function.add_model(model_name, model_path=model_path, class_name="AddTenModel")
function.deploy()

In [ ]:
# An example of invoke:
function.invoke(f"/v2/models/{model_name}/infer", {"inputs": [20]})["outputs"][
    "outputs"
]

<u>Application creation</u>:</br>
First, create the flask server application.</br>
The application includes several different endpoints - to check its functionality:

In [ ]:
%%writefile flask_app_example.py

from flask import Flask
import requests
import mlrun

app = Flask(__name__)


@app.route("/internal")
def internal():
    # Test access to the serving function with MLRun.
    project_name = "app-demo-flask"
    project = mlrun.get_or_create_project(project_name)
    function = project.get_function("add-ten-serving",ignore_cache=True)
    response = function.invoke("/v2/models/add-ten-model/infer", {"inputs": [20]})
    output = response["outputs"]["outputs"]
    return {"result": output}


@app.route("/external")
def external():
    # Test access to the serving function without MLRun (externally).
    project_name = "app-demo-flask"
    project = mlrun.get_or_create_project(project_name)
    function = project.get_function("add-ten-serving",ignore_cache=True) 
    url = f"https://{function.status.external_invocation_urls[0]}"
    response = requests.post(url, json={"inputs": [50]}).json()
    output = response["outputs"]["outputs"]
    return {"result": output}

Archive the application code into a .tar.gz file:

In [ ]:
!tar -czvf archive.tar.gz flask_app_example.py

Now, deploy this flask application:

In [ ]:
import mlrun

project = mlrun.get_or_create_project("app-demo-flask")
# Create a demo secret for testing
project.set_secrets(secrets={"secret-example": "project_secret_example"})

In [ ]:
# Specify source to be loaded on build time
# The image or the requirements should include mlrun, flask and gunicorn.
application = project.set_function(
    name="flask_app",
    kind="application",
    requirements_file="/your/path/requirements.txt",  # if needed
)

In [ ]:
# Provide code artifacts
application.with_source_archive(
    "v3io:///your/path/archive.tar.gz", pull_at_runtime=False
)

In [ ]:
application.set_internal_application_port(5000)
application.spec.command = "gunicorn"
application.spec.args = [
    "flask_app_example:app",
    "--bind",
    "127.0.0.1:5000",
    "--log-level",
    "debug",
]
application.deploy(with_mlrun=True)

In [5]:
# Test the deployment:
application.invoke("/internal").json()

In [ ]:
application.invoke("/external").json()

## Configure sidecar Kubernetes probes

You can configure Kubernetes liveness, readiness, and startup probes for the sidecar container using the {py:meth}`mlrun.runtimes.ApplicationRuntime.set_probe` and {py:meth}`mlrun.runtimes.ApplicationRuntime.delete_probe`  methods.
For more information, see the [Kubernetes Configure Probes] documentation(https://kubernetes.io/docs/tasks/configure-pod-container/configure-liveness-readiness-startup-probes/#configure-probes).

### Set probes

The `set_probe()` method allows you to configure probes.

Basic HTTP probe example:

In [ ]:
application.set_probe(
    type="liveness",
    http_path="/health",
    http_port=8080,
    http_scheme="HTTPS",
    initial_delay_seconds=15,
    period_seconds=10,
    failure_threshold=3,
    timeout_seconds=5,
)

For probe types not covered by the simple parameters (e.g., TCP socket, exec, or gRPC probes), you can use the `config` parameter to provide a full Kubernetes probe configuration:

In [ ]:
application.set_probe(
    type="liveness",
    initial_delay_seconds=15,
    period_seconds=10,
    config={
        "tcpSocket": {"port": 8080},
        "terminationGracePeriodSeconds": 90,
    },
)

### Usage
**Parameter precedence:** When both `config` and explicit parameters are provided, the explicit parameters override values in the config. Example:

In [ ]:
application.set_probe(
    type="startup",
    initial_delay_seconds=15,  # This will be used (not 20 from config)
    config={
        "tcpSocket": {"port": 8080},
        "initialDelaySeconds": 20,  # Overridden by parameter above
        "periodSeconds": 30,
    },
)

**Port enrichment:** If you provide `http_path` without `http_port`, the port will be automatically enriched from the `internal_application_port` just before deployment. Example:

In [ ]:
application.set_internal_application_port(8050)
application.set_probe(
    type="readiness",
    http_path="/health",
    # Port will be enriched to 8050 before deployment
)

**Set override:** Calling the `set_probe` method repeatedly with the same probe type will replace the existing configuration.

In [ ]:
application.set_probe(
    type="liveness",
    initial_delay_seconds=15,
    http_path="/health",
)

application.set_probe(type="liveness", http_path="/health", period_seconds=10)

# The liveness probe configuration will now only have period_seconds=10

**Health checks:** Each probe must define exactly one health check handler (`httpGet`, `tcpSocket`, `grpc` or `exec`) as required by Kubernetes; this rule is enforced defensively to ensure configuration compliance.

### Delete probes

To remove a probe configuration, use the `delete_probe()` method:

In [ ]:
application.delete_probe(type="readiness")